# COMP 365 Build 3 — API Tests
Cloud Deployment Readiness Advisor

Replace `SPACE_ID` with your deployed Hugging Face Space.

In [ ]:
%pip -q install -U gradio_client

from gradio_client import Client
SPACE_ID = "YOUR_USERNAME/YOUR_SPACE"
client = Client(SPACE_ID)
client.view_api()

In [ ]:
def call_case(**overrides):
    case = dict(
        deployment_environment="Production",
        tests_passed="True",
        test_coverage=90,
        secret_scan_passed="True",
        secrets_detected="False",
        environment_variables_configured="True",
        health_check_configured="True",
        database_migrations_ready="True",
        rollback_plan_available="True",
        monitoring_configured="True",
    )
    case.update(overrides)

    output = client.predict(
        deployment_environment=case["deployment_environment"],
        tests_passed=case["tests_passed"],
        test_coverage=case["test_coverage"],
        secret_scan_passed=case["secret_scan_passed"],
        secrets_detected=case["secrets_detected"],
        environment_variables_configured=case["environment_variables_configured"],
        health_check_configured=case["health_check_configured"],
        database_migrations_ready=case["database_migrations_ready"],
        rollback_plan_available=case["rollback_plan_available"],
        monitoring_configured=case["monitoring_configured"],
        api_name="/analyze",
    )
    # Gradio returns (markdown_explanation, structured_json)
    return output[1] if isinstance(output, (list, tuple)) else output

def rule_ids(result):
    return {item["id"] for item in result["rules_fired"]}

TESTS = []

def check(name, fn):
    try:
        fn()
        TESTS.append((name, True, ""))
        print("PASS", name)
    except Exception as e:
        TESTS.append((name, False, str(e)))
        print("FAIL", name, "-", e)


In [ ]:
check("1 Production conclusion + chained inference", lambda: (
    (lambda r: (
        (_ for _ in ()).throw(AssertionError(r)) if not (
            r["supported_conclusion"] == "READY_FOR_PRODUCTION"
            and {"R1","R2","R3","R4","R5","R6","R7"} <= rule_ids(r)
            and r["intermediate_facts"]["technical_readiness"] is True
        ) else None
    ))(call_case())
))

check("2 Staging conclusion", lambda: (
    (lambda r: (_ for _ in ()).throw(AssertionError(r))
     if not (r["supported_conclusion"] == "READY_FOR_STAGING" and "R8" in rule_ids(r))
     else None)(call_case(deployment_environment="Staging"))
))

check("3 Failed tests blocker", lambda: (
    (lambda r: (_ for _ in ()).throw(AssertionError(r))
     if not (r["supported_conclusion"] == "BLOCK_DEPLOYMENT" and "R10" in rule_ids(r))
     else None)(call_case(tests_passed="False"))
))

check("4 Secret blocker", lambda: (
    (lambda r: (_ for _ in ()).throw(AssertionError(r))
     if not (r["supported_conclusion"] == "BLOCK_DEPLOYMENT" and "R9" in rule_ids(r))
     else None)(call_case(secret_scan_passed="False", secrets_detected="True"))
))

check("5 Production migration blocker", lambda: (
    (lambda r: (_ for _ in ()).throw(AssertionError(r))
     if not (r["supported_conclusion"] == "BLOCK_DEPLOYMENT" and "R11" in rule_ids(r))
     else None)(call_case(database_migrations_ready="False"))
))

check("6 Unknown information is not False", lambda: (
    (lambda r: (_ for _ in ()).throw(AssertionError(r))
     if not (
        r["supported_conclusion"] is None
        and "tests_passed" in r["missing_information"]
        and "R10" not in rule_ids(r)
     ) else None)(call_case(tests_passed="Unknown"))
))

check("7 Conflicting information", lambda: (
    (lambda r: (_ for _ in ()).throw(AssertionError(r))
     if not (len(r["conflicts"]) > 0 and r["supported_conclusion"] == "BLOCK_DEPLOYMENT")
     else None)(call_case(secret_scan_passed="True", secrets_detected="True"))
))

check("8 Invalid input", lambda: (
    (lambda r: (_ for _ in ()).throw(AssertionError(r))
     if not (r["status"] == "invalid" and r["supported_conclusion"] is None)
     else None)(call_case(test_coverage=120))
))

check("9 No supported conclusion", lambda: (
    (lambda r: (_ for _ in ()).throw(AssertionError(r))
     if not (r["supported_conclusion"] is None and "R1" not in rule_ids(r))
     else None)(call_case(test_coverage=70))
))

check("10 Deploy with caution", lambda: (
    (lambda r: (_ for _ in ()).throw(AssertionError(r))
     if not (
         r["supported_conclusion"] == "DEPLOY_WITH_CAUTION"
         and r["intermediate_facts"]["technical_readiness"] is True
     ) else None)(call_case(rollback_plan_available="Unknown"))
))

first = call_case()
second = call_case()
check("11 Determinism", lambda: (
    None if first == second else (_ for _ in ()).throw(AssertionError("Outputs differ"))
))


In [ ]:
import pandas as pd

rows = [{"test": n, "passed": p, "detail": d} for n, p, d in TESTS]
display(pd.DataFrame(rows))

# Rule coverage from scenarios that intentionally trigger every rule.
coverage_cases = [
    call_case(),
    call_case(deployment_environment="Staging"),
    call_case(tests_passed="False"),
    call_case(secret_scan_passed="False", secrets_detected="True"),
    call_case(database_migrations_ready="False"),
]
covered = set()
for r in coverage_cases:
    covered |= rule_ids(r)

all_rules = {f"R{i}" for i in range(1, 12)}
missing = sorted(all_rules - covered)

print(f"Tests passed: {sum(p for _, p, _ in TESTS)}/{len(TESTS)}")
print(f"Rule coverage: {len(covered)}/{len(all_rules)}")
print("Covered:", sorted(covered))
print("Missing:", missing)

assert all(p for _, p, _ in TESTS), "At least one API test failed."
assert not missing, f"Missing rule coverage: {missing}"
print("\nALL API TESTS PASSED AND ALL RULES ARE COVERED.")
